In [0]:
import requests
from pyspark.sql import functions as F

url = "https://data.cityofnewyork.us/resource/m6nq-qud6.json"

params = {
    "$limit":  50000,
    "$offset": 0
}

response = requests.get(url, params=params)
data     = response.json()

print(f"Records fetched: {len(data):,}")

# Load into Spark DataFrame
df_newyorkdata = spark.createDataFrame(data)
col = F.concat_ws( *[F.col(c) for c in df_newyorkdata.columns])



**Adding lineage columns **

In [0]:
df_newyorkdata_count =  df_newyorkdata.count()
df_lineage_columns =(df_newyorkdata.withColumn("_loaddatetime",F.current_timestamp())
                                    .withColumn("_batchid",F.date_format(F.current_date(),"yyyyMMdd")  )
                                          
                                    .withColumn("_source_system", F.lit("nyc_taxi_api"))
                                    .withColumn("_source_url", F.lit("data.cityofnewyork.us/resource/m6nq-qud6"))
                                    .withColumn("_load_count",F.lit(df_newyorkdata_count))
                                    .withColumn("_hashid",F.md5(col))
                    )


**Loading raw data into a Bronze-layer Delta table.**


In [0]:
# Cmd 1 — Create schema first
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze_layer")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver_layer")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold_layer")

# Cmd 2 — Write directly as managed table
df_lineage_columns.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.bronze_layer.nyc_taxi_yellow_raw")

# Cmd 3 — Verify
spark.sql("SELECT COUNT(*) FROM workspace.bronze_layer.nyc_taxi_yellow_raw").show()

In [0]:
%sql

--SHOW VOLUMES IN workspace.default;

In [0]:
%sql
--CREATE VOLUME IF NOT EXISTS workspace.default.bronze;
--CREATE VOLUME IF NOT EXISTS workspace.default.silver;
--CREATE VOLUME IF NOT EXISTS workspace.default.gold;